# Reshaping metadata

Source files with medatata providing via different files. Some data are privacy sensitive (f.e. mapping KYH study ID with 16S sequencing library id ) 

So I aggregate required field into CSV files

- `kyh_metadata.csv` contains open clinical and personal data from KYH study
- `kyh_variables_description.csv` human readable descriptions for columns names from kyh_metadata.csv
- `kyh_ids_map.csv` _private_ mapping KYH study ID with 16S sequencing library id 
- `SraRunTable.csv` intact sra accesses from NCBI SRA Run Selector

`all_meta.csv` _private_ kyh_metadata.csv with Run Id from SraRunTable.csv (mapped via kyh_ids_map.csv)

__Git repo contains these ready to use metadata files__ (except marked as _private_)

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
METADATA_DIR = Path("data/initial")

## Load metadata sets

In [3]:
kyh_df = pd.read_stata(METADATA_DIR / "260505_Alex_Zobolev.dta", index_col='KYH_ID')
kyh_df = kyh_df.replace({'-999': pd.NA})
kyh_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2380 entries, 1121127 to 1029104
Data columns (total 55 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   hcd_date              2380 non-null   datetime64[ns]
 1   d_sex                 2380 non-null   category      
 2   d_age_hc              2380 non-null   float64       
 3   d_age_hc_grp_10yr     2380 non-null   category      
 4   d_age_hc_grp_5yr      2380 non-null   category      
 5   b_a9                  2380 non-null   category      
 6   bd_education3_vs      2380 non-null   category      
 7   bd_phq9_sev_score     2380 non-null   int8          
 8   bd_phq9_dep_alt       2380 non-null   category      
 9   bd_phq9_symptomcount  2380 non-null   int8          
 10  bd_phq9_dep_cat       2380 non-null   category      
 11  bd_phq9_any_dep       2380 non-null   category      
 12  bd_phq9_mod_dep       2380 non-null   category      
 13  b_e1          

In [4]:
kyh_variables = pd.read_excel(METADATA_DIR / 'Ark_Dataset_description.xlsx', index_col=0)
kyh_variables = pd.concat([kyh_variables, pd.DataFrame.from_dict(
    {f'atc{n}': f'ATC group coding of medication {n}' for n in range(1, 8) },
    columns=['Label'],
    orient='index'
)])

In [5]:
sra_table = pd.read_csv(METADATA_DIR / "SraRunTable.csv", index_col="Sample Name")
sra_table.head()

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,Library Name,LibraryLayout,LibrarySelection,LibrarySource,Organism,Platform,ReleaseDate,create_date,version,SRA Study
Sample Name,,,,,,,,,,,,,,,,,,,,,
KYH512155,SRR32393053,AMPLICON,500,23148500,PRJNA1225157,SAMN46890594,"MIMS.me,MIGS/MIMS/MIMARKS.human-gut",13457009,RESEARCH INSTITUTE FOR SYSTEMS BIOLOGY AND MED...,2016-09-06/2017-11-27,...,KYH512155,PAIRED,PCR,METAGENOMIC,human gut metagenome,ILLUMINA,2025-03-07T00:00:00Z,2025-02-19T06:22:00Z,1,SRP564871
KYH950050,SRR32393054,AMPLICON,501,13156273,PRJNA1225157,SAMN46891187,"MIMS.me,MIGS/MIMS/MIMARKS.human-gut",8161868,RESEARCH INSTITUTE FOR SYSTEMS BIOLOGY AND MED...,2016-09-06/2017-11-27,...,KYH950050,PAIRED,PCR,METAGENOMIC,human gut metagenome,ILLUMINA,2025-03-07T00:00:00Z,2025-02-19T06:22:00Z,1,SRP564871
KYH337127,SRR32393055,AMPLICON,501,16897469,PRJNA1225157,SAMN46891186,"MIMS.me,MIGS/MIMS/MIMARKS.human-gut",9919188,RESEARCH INSTITUTE FOR SYSTEMS BIOLOGY AND MED...,2016-09-06/2017-11-27,...,KYH337127,PAIRED,PCR,METAGENOMIC,human gut metagenome,ILLUMINA,2025-03-07T00:00:00Z,2025-02-19T06:22:00Z,1,SRP564871
KYH643147,SRR32393056,AMPLICON,501,16019213,PRJNA1225157,SAMN46891185,"MIMS.me,MIGS/MIMS/MIMARKS.human-gut",10286511,RESEARCH INSTITUTE FOR SYSTEMS BIOLOGY AND MED...,2016-09-06/2017-11-27,...,KYH643147,PAIRED,PCR,METAGENOMIC,human gut metagenome,ILLUMINA,2025-03-07T00:00:00Z,2025-02-19T06:22:00Z,1,SRP564871
KYH619523,SRR32393057,AMPLICON,501,21237212,PRJNA1225157,SAMN46891184,"MIMS.me,MIGS/MIMS/MIMARKS.human-gut",12225672,RESEARCH INSTITUTE FOR SYSTEMS BIOLOGY AND MED...,2016-09-06/2017-11-27,...,KYH619523,PAIRED,PCR,METAGENOMIC,human gut metagenome,ILLUMINA,2025-03-07T00:00:00Z,2025-02-19T06:22:00Z,1,SRP564871


In [6]:
ids_map = pd.read_csv(METADATA_DIR / "old_metadata_with_ids_mappings.csv", sep=';', index_col='ID_KYH')
ids_map.head()

,df_sample,Unnamed: 0,batch,batch2,rep,ID_Dima,ID_David,main_group,site,date_of_brth,...,high_hba1c,diabetes_medical,drinking_level,ISCO_group,anti_inflammatory_med,PPI_med,antibiotics_med,lipid_mod_agents_med,antidiab_med,new_id
ID_KYH,,,,,,,,,,,,,,,,,,,,,
1084344,1084344_B1,1084344_B1,B1,B1,NaN,1,1,Arkhangelsk main study,Arkhangelsk,18.06.1950,...,yes,yes,Non-problem drinkers,High skilled white collar (ISCO codes 1-3),no,no,no,no,no,KYH272685
1068937,1068937_B1,1068937_B1,B1,B1,NaN,1,1,Arkhangelsk main study,Arkhangelsk,11.07.1968,...,no,no,Non-problem drinkers,High skilled blue collar (ISCO codes 6-7),no,no,no,no,no,KYH879167
1152946,1152946_B1,1152946_B1,B1,B1,NaN,1,1,Arkhangelsk main study,Arkhangelsk,25.11.1974,...,no,no,Non-problem drinkers,High skilled white collar (ISCO codes 1-3),no,no,no,no,no,KYH518700
1127440,1127440_B1,1127440_B1,B1,B1,NaN,1,1,Arkhangelsk main study,Arkhangelsk,17.10.1973,...,no,no,Non-problem drinkers,High skilled white collar (ISCO codes 1-3),no,no,no,no,no,KYH444549
1118366,1118366_B1,1118366_B1,B1,B1,NaN,1,1,Arkhangelsk main study,Arkhangelsk,11.04.1956,...,no,no,Non-problem drinkers,Low skilled white collar (ISCO codes 4-5),no,no,no,no,no,KYH552241


## Map human readable descriptions for metadata

In [7]:
print(kyh_df.columns[ ~ kyh_df.columns.isin(kyh_variables.index)])
kyh_variables = kyh_variables.loc[kyh_df.columns[kyh_df.columns.isin(kyh_variables.index)]]
kyh_variables

Index([], dtype='object')


,Label
hcd_date,Date of Health Check
d_sex,Derived sex
d_age_hc,Derived age at heallth check
d_age_hc_grp_10yr,Derived age at health check 10yr grp
d_age_hc_grp_5yr,Derived age at health check 5yr grp
b_a9,Education level
bd_education3_vs,Education 3 categories
bd_phq9_sev_score,PHQ-9 Depression severity score
bd_phq9_dep_alt,PHQ-9 depression alternative scoring
bd_phq9_symptomcount,PHQ-9 symptom count


In [8]:
kyh_variables.to_csv(METADATA_DIR / "kyh_variables_description.csv")
kyh_variables = pd.read_csv(METADATA_DIR / "kyh_variables_description.csv", index_col=0)

## Filter KYH metadata 

Select KYH metadata by 16s sequencing library ids

### Find out are there unmatched indexes

In [9]:
print('sra_table, ids_map:', len(sra_table.index.symmetric_difference(ids_map['new_id'])))
print('kyh_df, ids_map:', len(ids_map.index[ ~ids_map.index.isin(kyh_df.index)]))

sra_table, ids_map: 0
kyh_df, ids_map: 0


In [10]:
kyh_df = kyh_df.loc[ids_map.index]

## Medications 

Discover the reception of __antibiotics__ (ATC code `J01`) or __antidepressants__ (ATC code `N06A`)

In [11]:
atc_cols = [f'atc{i}' for i in range(1, 8)]
def isMedicationUsage(df, ATCCode, new_col_name) -> pd.DataFrame:
    df[new_col_name] = (
        df[atc_cols]
        .astype(str)
        .apply(lambda col: col.str.startswith(ATCCode, na=False))
        .any(axis=1)
    )
    return df

kyh_df = isMedicationUsage(kyh_df, 'N06A', 'antidepressant_usage')
kyh_df = isMedicationUsage(kyh_df, 'J01', 'antibiotics_usage')
kyh_df[['antidepressant_usage', 'antibiotics_usage']].sum()

antidepressant_usage    0
antibiotics_usage       2
dtype: int64

Filter out patient taken antibiotics (2 samples) and antidepressant_usage (0 samples)

In [12]:
kyh_df = kyh_df[~ kyh_df[['antidepressant_usage', 'antibiotics_usage']].any(axis=1)]

## Depression summary

In [13]:
for col in kyh_df.loc[:,'bd_phq9_sev_score':'bd_phq9_mod_dep']:
    print(kyh_df[col].value_counts(dropna=False), '\n')

bd_phq9_sev_score
0     145
3      97
1      84
4      81
2      68
5      62
6      42
7      28
9      20
8      15
10     10
13      7
11      7
12      5
15      5
16      3
14      1
17      1
18      1
25      1
Name: count, dtype: int64 

bd_phq9_dep_alt
no depression    626
depression        57
Name: count, dtype: int64 

bd_phq9_symptomcount
0    145
3    119
1    105
2     93
4     83
5     72
6     30
7     25
8      9
9      2
Name: count, dtype: int64 

bd_phq9_dep_cat
less than 5 no depression                   475
5-9 Mild depression                         167
10-14 moderate depression                    30
15-19 major depression moderately severe     10
>20 Major depression, severe                  1
Name: count, dtype: int64 

bd_phq9_any_dep
PHQ-9<5     475
PHQ-9>=5    208
Name: count, dtype: int64 

bd_phq9_mod_dep
PHQ-9<10     642
PHQ-9>=10     41
Name: count, dtype: int64 



## Season

Compare _Date faecal sample was collected by participant_ (`lab_lc_w8b`) vs. _Date of Health Check_ (`hcd_date`) 

If `lab_lc_w8b` is empty, i will use `hcd_date` instead

In [14]:
kyh_df['lab_lc_w8b'] = kyh_df['lab_lc_w8b'].replace({'97': pd.NA}) # i really don't know what 97 means, but it is not a valid date, so I will treat it as missing value
kyh_df['lab_lc_w8b'] = pd.to_datetime(kyh_df['lab_lc_w8b'], format='mixed')

In [15]:
season = ['cold', 'cold', 'warm', 'warm', 'warm', 'warm', 'warm', 'warm', 'cold', 'cold', 'cold', 'cold']
kyh_df['season'] = kyh_df['lab_lc_w8b'].fillna(kyh_df['hcd_date']).dt.month.apply(lambda x: season[int(x)-1])

In [16]:
table = kyh_df.pivot_table(
    index='bd_phq9_any_dep',
    columns='season',
    aggfunc='size',
    fill_value=0,
    observed=True 
)
table

season,cold,warm
bd_phq9_any_dep,,
PHQ-9<5,256,219
PHQ-9>=5,101,107


In [17]:
from scipy.stats import chi2_contingency
pd.DataFrame(chi2_contingency(table), index=['Chi2 statistic', 'p-value', 'degrees of freedom', 'expected frequencies'])

,0
Chi2 statistic,1.444561
p-value,0.229403
degrees of freedom,1
expected frequencies,"[[248.2796486090776, 226.7203513909224], [108...."


## Save data

In [18]:
kyh_df.to_csv(METADATA_DIR / "kyh_metadata.csv")
ids_map = ids_map['new_id']
ids_map.to_csv(METADATA_DIR / "kyh_ids_map.csv")

#### File with all needed metadate (_private_)

In [19]:
kyh_df['new_id'] = kyh_df.index.map(ids_map)
kyh_df = kyh_df.merge(
    sra_table[['Run']],
    left_on='new_id',
    right_index=True,
    how='left'
)
kyh_df.to_csv(METADATA_DIR / "all_meta.csv")